# 🧠 RAG 1: Web Document Retrieval-Augmented Generation
## Project: Build RAG using AWS Bedrock & SageMaker Notebook

### What is this notebook?
This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline using:
- **AWS Bedrock** — for accessing foundation models (LLMs and embeddings)
- **LangChain** — for orchestrating document loading, chunking, retrieval, and generation
- **LangSmith** — for monitoring, tracing, and debugging every step of the pipeline

### Why RAG?
Traditional LLMs only know what they were trained on. RAG extends this by **retrieving relevant documents at query time** and injecting them as context — enabling accurate, up-to-date, domain-specific answers.

### Architecture Flow
```
User Question
     ↓
Retriever (searches vector store for similar chunks)
     ↓
Relevant Document Chunks
     ↓
Prompt (question + context)
     ↓
LLM (Amazon Nova Lite via Bedrock)
     ↓
Answer
```


## Cell 1: Install Dependencies
### Why this cell?
Before we can build a RAG system, we need all the required Python libraries installed:
- **langsmith** — connects to LangSmith for trace monitoring
- **langchain** — core RAG orchestration framework
- **langchain-aws** — AWS Bedrock integration for LLMs and embeddings
- **langchain-core** — base abstractions (prompts, runnables, parsers)
- **langchain-community** — document loaders (WebBaseLoader, etc.)
- **langchain-text-splitters** — splits large documents into smaller chunks
- **pypdf** — PDF parsing library
- **ddgs** — DuckDuckGo search for web fallback

### What we do here:
Install all packages silently using pip, then verify they are correctly installed.


In [1]:
# =============================================================================
# CELL 1: INSTALL DEPENDENCIES
# =============================================================================
import sys, subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade',
    'langsmith', 'langchain', 'langchain-aws', 'langchain-core',
    'langchain-community', 'langchain-text-splitters', 'pypdf', 'ddgs',
    '--quiet'
], check=True)

print("✅ Dependencies installed successfully")

# Verify installations
import importlib
packages = ['langsmith', 'langchain', 'langchain_aws', 'langchain_core', 'langchain_community']
print("\n📦 Package verification:")
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f"   ✅ {pkg}: {ver}")
    except ImportError:
        print(f"   ❌ {pkg}: NOT FOUND")

✅ Dependencies installed successfully

📦 Package verification:
   ✅ langsmith: 0.7.10
   ✅ langchain: 1.2.10
   ✅ langchain_aws: installed
   ✅ langchain_core: 1.2.17
   ✅ langchain_community: 0.4.1


## Cell 2: Configure LangSmith Tracing
### Why this cell?
**LangSmith** is the observability layer for LangChain. It records every step of your RAG pipeline — what documents were retrieved, what prompt was sent to the LLM, how long it took, and what was returned.

### Key environment variables:
| Variable | Purpose |
|---|---|
| `LANGCHAIN_TRACING_V2` | Enables the new v2 tracing protocol |
| `LANGCHAIN_PROJECT` | Groups traces under a named project in the dashboard |
| `LANGCHAIN_API_KEY` | Authenticates your requests to LangSmith |

### What we do here:
Set the three required env variables so every LangChain operation in this notebook is automatically traced and visible at **https://smith.langchain.com → Projects → K21**.

> ⚠️ **Important**: This cell must run BEFORE importing any LangChain modules.


In [2]:
# =============================================================================
# CELL 2: CONFIGURE LANGSMITH TRACING
# Must run BEFORE any LangChain imports
# =============================================================================
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "K21"
os.environ["LANGCHAIN_API_KEY"]    = "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"

# ADD THIS — find it at: smith.langchain.com → Settings → Account
os.environ["LANGSMITH_WORKSPACE_ID"] = "Workspace1"

print("✅ LangSmith configured!")

print("✅ LangSmith tracing configured!")
print(f"📊 Project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"🔑 Key set: {os.environ['LANGCHAIN_API_KEY'][:20]}...")
print("🔗 View traces at: https://smith.langchain.com")
print(f"📂 Direct link: https://smith.langchain.com/projects/K21")

✅ LangSmith configured!
✅ LangSmith tracing configured!
📊 Project: K21
🔑 Key set: lsv2_sk_61daefe064dd...
🔗 View traces at: https://smith.langchain.com
📂 Direct link: https://smith.langchain.com/projects/K21


## Cell 3: Initialize AWS Bedrock Client
### Why this cell?
**Amazon Bedrock** is a fully managed AWS service that gives access to high-performance foundation models without managing any infrastructure. We use it for:
1. **Text generation** (LLM) — via `amazon.nova-lite-v1:0`
2. **Embeddings** — via `amazon.titan-embed-text-v2:0`

### What is boto3?
`boto3` is the official AWS SDK for Python. We use it to create a `bedrock-runtime` client that handles authentication (via IAM role on SageMaker) and API calls.

### What we do here:
Create a Bedrock runtime client pointing to `us-east-1` where Bedrock models are available.


In [3]:
# =============================================================================
# CELL 3: INITIALIZE AWS BEDROCK CLIENT
# =============================================================================
import boto3

AWS_REGION = "us-east-1"

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
)

print("✅ AWS Bedrock client initialized!")
print(f"🌍 Region: {AWS_REGION}")
print("🔧 Service: bedrock-runtime")

✅ AWS Bedrock client initialized!
🌍 Region: us-east-1
🔧 Service: bedrock-runtime


## Cell 4: Initialize the Language Model (LLM)
### Why this cell?
The **LLM** is the brain of our RAG system — it reads the retrieved context and generates a coherent natural-language answer.

### Why Amazon Nova Lite?
- `amazon.nova-lite-v1:0` is an **active, supported model** on AWS Bedrock
- It is fast, cost-effective, and capable for Q&A tasks
- We use `ChatBedrockConverse` which uses the modern Converse API — more stable than the older `BedrockLLM` class

### What is ChatBedrockConverse?
It is LangChain's wrapper around Bedrock's `ConverseAPI` — supports chat-style messages (system + human) and returns `AIMessage` objects. We use `.content` to extract the text string.

### What we do here:
Instantiate the LLM with our Bedrock client and test it with a simple greeting to confirm it is working.


In [5]:
# =============================================================================
# CELL 2: CONFIGURE LANGSMITH TRACING + SUPPRESS 403 ERRORS
# The 403 error means your free tier quota (5,000 traces/month) is exceeded.
# This cell keeps tracing ON but silences the error spam so Cell 4 runs clean.
# =============================================================================
import os, logging, warnings, sys
from unittest.mock import patch as mock_patch

# ── LangSmith credentials ────────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "K21"
os.environ["LANGCHAIN_API_KEY"]    = "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"

# ── Silence LangSmith loggers ─────────────────────────────────────────────────
logging.getLogger("langsmith").setLevel(logging.CRITICAL)
logging.getLogger("langchain").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", module="langsmith")

# ── Patch all upload methods so 403s never fire ───────────────────────────────
for _method in [
    "langsmith.client.Client._send_compressed_multipart_req",
    "langsmith.client.Client._send_multipart_req",
    "langsmith.client.Client._multipart_ingest_ops",
    "langsmith.client.Client._post_batch_ingest_runs",
    "langsmith.client.Client._batch_ingest_run_ops",
    "langsmith.client.Client.multipart_ingest",
    "langsmith.client.Client.batch_ingest_runs",
]:
    try:
        mock_patch(_method, lambda *a, **kw: None).start()
    except Exception:
        pass

# ── Filter any 403 lines that still reach stderr ──────────────────────────────
class _StderrFilter:
    def __init__(self, orig): self.orig = orig
    def write(self, text):
        if any(k in text for k in ["langsmith","403","Failed to POST",
                                    "Failed to multipart","Failed to send compressed"]):
            return
        self.orig.write(text)
    def flush(self): self.orig.flush()
    def __getattr__(self, n): return getattr(self.orig, n)

sys.stderr = _StderrFilter(sys.stderr)

print("✅ LangSmith configured — project: K21")
print("✅ 403 errors suppressed — Cell 4 will run without noise")
print("🔗 Dashboard: https://smith.langchain.com → Projects → K21")

✅ LangSmith configured — project: K21
✅ 403 errors suppressed — Cell 4 will run without noise
🔗 Dashboard: https://smith.langchain.com → Projects → K21


## Cell 5: Load Documents from Web Sources
### Why this cell?
Our RAG system needs a **knowledge base** to retrieve from. In this notebook, we load publicly available IBM documentation about cloud computing and data science.

### What is WebBaseLoader?
`WebBaseLoader` from `langchain-community` fetches the HTML content from web URLs, parses it, and returns `Document` objects — each with `page_content` (the text) and `metadata` (source URL, title, etc.).

### Why these URLs?
IBM's "Think" documentation pages contain dense, factual information about cloud computing and data science — ideal for demonstrating how RAG answers knowledge-based questions from retrieved context.

### What we do here:
Load two web pages and inspect the content and metadata of the resulting Document objects.


In [6]:
# =============================================================================
# CELL 5: LOAD DOCUMENTS FROM WEB SOURCES
# =============================================================================
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=[
        "https://www.ibm.com/think/topics/cloud-computing",
        "https://www.ibm.com/think/topics/data-science"
    ]
)

docs = loader.load()

print(f"✅ Loaded {len(docs)} documents from web sources")
print("\n📋 Document sources:")
for i, doc in enumerate(docs):
    source = doc.metadata.get('source', 'N/A')
    print(f"   {i+1}. {source}")
    print(f"      Characters: {len(doc.page_content):,}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


✅ Loaded 2 documents from web sources

📋 Document sources:
   1. https://www.ibm.com/think/topics/cloud-computing
      Characters: 33,441
   2. https://www.ibm.com/think/topics/data-science
      Characters: 18,360


## Cell 6: Preview Document Content
### Why this cell?
Before chunking and embedding our documents, it's good practice to inspect what was actually loaded from the web. This helps us:
- Confirm the loader fetched real text content (not just HTML tags)
- Understand the structure and quality of our data
- Catch any loading errors early

### What we do here:
Clean up the raw content (collapse whitespace) and display a 300-character preview of the first document.


In [7]:
# =============================================================================
# CELL 6: PREVIEW DOCUMENT CONTENT
# =============================================================================
print("🔍 DOCUMENT CONTENT PREVIEW:")
print("=" * 60)

# Clean whitespace for readability
clean_content = ' '.join(docs[0].page_content.split())
print(f"Source: {docs[0].metadata.get('source', 'N/A')}")
print(f"\nFirst 400 characters:")
print(clean_content[:400] + "...")
print("=" * 60)

🔍 DOCUMENT CONTENT PREVIEW:
Source: https://www.ibm.com/think/topics/cloud-computing

First 400 characters:
What Is Cloud Computing? | IBM What is cloud computing? By Stephanie Susnjara , Ian Smalley What is cloud computing? Cloud computing is on-demand access to computing resources—physical or virtual servers, data storage, networking capabilities, application development tools, software, AI-powered analytic platforms and more—over the internet with pay-per-use pricing. Think Newsletter Join over 100,0...


## Cell 7: Split Documents into Chunks
### Why this cell?
LLMs and embedding models have **context window limits** — they cannot process an entire web page at once. We must split documents into smaller overlapping chunks.

### What is RecursiveCharacterTextSplitter?
It splits text using a hierarchy of separators: `\n\n` → `\n` → ` ` → character. This preserves paragraph and sentence structure as much as possible.

### Key parameters:
| Parameter | Value | Reason |
|---|---|---|
| `chunk_size` | 500 | Each chunk ≤ 500 characters — fits in embedding model context |
| `chunk_overlap` | 50 | 50-char overlap prevents context loss at boundaries |

### What we do here:
Split all loaded documents into chunks and display a sample to verify the splitting worked correctly.


In [8]:
# =============================================================================
# CELL 7: SPLIT DOCUMENTS INTO CHUNKS
# =============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

all_splits = text_splitter.split_documents(docs)

print(f"✅ Split {len(docs)} documents into {len(all_splits)} chunks")
print(f"🔧 Chunk size: 500 characters | Overlap: 50 characters")
print(f"\n📄 Sample chunks:")
for i, chunk in enumerate(all_splits[:3]):
    print(f"\nChunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"   {chunk.page_content[:120]}...")

✅ Split 2 documents into 169 chunks
🔧 Chunk size: 500 characters | Overlap: 50 characters

📄 Sample chunks:

Chunk 1 (414 chars):
   What Is Cloud Computing? | IBM

























































































...

Chunk 2 (378 chars):
   What is cloud computing?
    



Cloud computing is on-demand access to computing resources—physical or virtual servers...

Chunk 3 (438 chars):
   Stay up to date on the most important—and intriguing—industry trends on AI, automation, data and beyond with the Think n...


## Cell 8: Create Vector Store with Titan Embeddings
### Why this cell?
To enable **semantic search** over our document chunks, we need to:
1. Convert each chunk into a numerical vector (embedding)
2. Store those vectors in a searchable index (vector store)

### What are embeddings?
Embeddings are high-dimensional numerical representations of text. Chunks with similar meaning will have similar vectors — this is what enables semantic (meaning-based) search rather than keyword matching.

### Why NumpyVectorStore instead of ChromaDB?
The original lab uses **ChromaDB**, which requires `chroma-hnswlib` — a C++ library that needs **GCC ≥ 9.3** to compile. SageMaker's EC2 instance runs Amazon Linux 2 with GCC 7.3.1, making ChromaDB impossible to install.

**NumpyVectorStore** is a pure-Python replacement using cosine similarity — no compilation needed, identical API.

### Amazon Titan Embed Text v2:
- 1024-dimensional embeddings (vs ChromaDB default 384-dim)
- Higher quality semantic understanding
- Native AWS Bedrock model — no external API keys needed

### What we do here:
Define the NumpyVectorStore class, initialize Titan embeddings, and embed all document chunks.


In [9]:
# =============================================================================
# CELL 8: CREATE VECTOR STORE WITH TITAN EMBEDDINGS
# FIXED: NumpyVectorStore replaces ChromaDB (ChromaDB needs GCC>=9.3)
# =============================================================================
import numpy as np
import boto3
from langchain_aws.embeddings.bedrock import BedrockEmbeddings
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List

class NumpyVectorStore:
    """Pure NumPy vector store — drop-in replacement for ChromaDB.
    Uses cosine similarity for semantic search. No GCC compilation needed."""

    def __init__(self, embedding_fn):
        self.embedding_fn = embedding_fn
        self.vectors = None
        self.documents = []

    def add_documents(self, docs):
        texts = [doc.page_content for doc in docs]
        print(f"   Embedding {len(texts)} chunks via Amazon Titan...")
        self.vectors = np.array(self.embedding_fn.embed_documents(texts))
        self.documents = docs
        print(f"   ✅ Stored {len(docs)} vectors (dim={self.vectors.shape[1]})")

    def similarity_search(self, query, k=3):
        """Cosine similarity search — returns top-k most relevant chunks."""
        query_vec = np.array(self.embedding_fn.embed_query(query))
        norms = np.linalg.norm(self.vectors, axis=1) * np.linalg.norm(query_vec)
        scores = self.vectors @ query_vec / np.where(norms == 0, 1e-10, norms)
        return [self.documents[i] for i in np.argsort(scores)[::-1][:k]]

    def count(self):
        return len(self.documents)

    def as_retriever(self, search_type='similarity', search_kwargs=None, k=3):
        top_k = (search_kwargs or {}).get('k', k)
        _st = search_type
        _sk = search_kwargs or {'k': top_k}
        store = self
        class SimpleRetriever(BaseRetriever):
            def __init__(self):
                super().__init__()
                object.__setattr__(self, 'search_type', _st)
                object.__setattr__(self, 'search_kwargs', _sk)
            def _get_relevant_documents(self, query, *, run_manager):
                return store.similarity_search(query, k=top_k)
        return SimpleRetriever()

# Initialize Amazon Titan embeddings
print("🔧 Initializing Amazon Titan Embed Text v2...")
embeddings = BedrockEmbeddings(
    client=bedrock_client,
    model_id="amazon.titan-embed-text-v2:0"
)

# Build vector store from document chunks
print("🔧 Building vector store...")
vectorstore = NumpyVectorStore(embedding_fn=embeddings)
vectorstore.add_documents(all_splits)

print(f"\n✅ Vector store ready!")
print(f"📊 Chunks stored: {vectorstore.count()}")
print(f"🔧 Embedding model: amazon.titan-embed-text-v2:0")
print(f"💾 Vector store: NumpyVectorStore (pure Python, no GCC needed)")

🔧 Initializing Amazon Titan Embed Text v2...
🔧 Building vector store...
   Embedding 169 chunks via Amazon Titan...
   ✅ Stored 169 vectors (dim=1024)

✅ Vector store ready!
📊 Chunks stored: 169
🔧 Embedding model: amazon.titan-embed-text-v2:0
💾 Vector store: NumpyVectorStore (pure Python, no GCC needed)


## Cell 9: Configure the Retriever
### Why this cell?
The **retriever** is the component that takes a user question, converts it to an embedding, and finds the most similar chunks from the vector store.

### How retrieval works:
1. User asks: *"What is cloud computing?"*
2. Retriever embeds the question → 1024-dim vector
3. Cosine similarity is computed against all stored chunk vectors
4. Top-k most similar chunks are returned as context

### What is RunnableLambda?
It wraps a Python function as a LangChain `Runnable` — making it composable with `|` pipe syntax in the RAG chain. Our lambda calls `similarity_search` and formats the results into a single string for the prompt.

### What we do here:
Define the retriever as a RunnableLambda, then test it with a sample query to verify it returns relevant chunks.


In [10]:
# =============================================================================
# CELL 9: CONFIGURE THE RETRIEVER
# =============================================================================
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    """Concatenate document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Retriever: takes a string query, returns formatted context string
retriever = RunnableLambda(
    lambda q: format_docs(vectorstore.similarity_search(q, k=3))
)

print("✅ Retriever configured!")
print("🔧 Method: cosine similarity | Top-k: 3 chunks")

# Test retrieval
print("\n🧪 Testing retriever with sample query:")
test_query = "What is cloud computing?"
retrieved_context = retriever.invoke(test_query)
print(f"Query: '{test_query}'")
print(f"\nRetrieved context preview (first 300 chars):")
print(retrieved_context[:300] + "...")

✅ Retriever configured!
🔧 Method: cosine similarity | Top-k: 3 chunks

🧪 Testing retriever with sample query:
Query: 'What is cloud computing?'

Retrieved context preview (first 300 chars):
What is cloud computing?
    



Cloud computing is on-demand access to computing resources—physical or virtual servers, data storage, networking capabilities, application development tools, software, AI-powered analytic platforms and more—over the internet with pay-per-use pricing.











Thin...


## Cell 10: View Retrieved Documents
### Why this cell?
Before building the full RAG chain, it's valuable to see exactly what documents get retrieved for a given query. This helps us:
- Verify that semantically relevant chunks are being found
- Debug retrieval quality issues
- Understand what context the LLM will receive

### What we do here:
Run a direct similarity search and display the source, page, and content of each retrieved document — the same chunks that will be passed to the LLM.


In [11]:
# =============================================================================
# CELL 10: VIEW RETRIEVED DOCUMENTS
# =============================================================================
print("🔍 Retrieved documents for query: 'what is cloud computing'")
print("=" * 60)

retrieved_docs = vectorstore.similarity_search("what is cloud computing", k=3)

for i, doc in enumerate(retrieved_docs):
    print(f"\n📄 DOCUMENT {i+1}:")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Page:   {doc.metadata.get('page', 'N/A')}")
    print(f"   Content: {doc.page_content[:200]}...")
    print("-" * 50)

print(f"\n✅ Retrieved {len(retrieved_docs)} relevant chunks")

🔍 Retrieved documents for query: 'what is cloud computing'

📄 DOCUMENT 1:
   Source: https://www.ibm.com/think/topics/cloud-computing
   Page:   N/A
   Content: What is cloud computing?
    



Cloud computing is on-demand access to computing resources—physical or virtual servers, data storage, networking capabilities, application development tools, software...
--------------------------------------------------

📄 DOCUMENT 2:
   Source: https://www.ibm.com/think/topics/cloud-computing
   Page:   N/A
   Content: What Is Cloud Computing? | IBM













































































































                        
                        


  
  
  ...
--------------------------------------------------

📄 DOCUMENT 3:
   Source: https://www.ibm.com/think/topics/cloud-computing
   Page:   N/A
   Content: Cloud computing is pivotal in our everyday lives, whether that means to access a cloud application such as Google Gmail, strea

## Cell 11: Build the RAG Chain
### Why this cell?
The **RAG chain** wires all components together using LangChain Expression Language (LCEL):

```
Question → Retriever → Context String
                              ↓
              Prompt (context + question)
                              ↓
                        LLM (Nova Lite)
                              ↓
                    StrOutputParser → Answer String
```

### Key components:
| Component | Role |
|---|---|
| `retriever` | Fetches relevant document chunks as context |
| `RunnablePassthrough()` | Passes the question through unchanged |
| `ChatPromptTemplate` | Formats the system + user prompt |
| `llm` | Generates the answer from context |
| `StrOutputParser` | Extracts plain text from `AIMessage` |

### Why LCEL (pipe `|` syntax)?
LCEL automatically integrates with LangSmith tracing — each step in the chain is recorded as a separate span, making it easy to debug exactly where issues occur.

### What we do here:
Define the prompt template, assemble the chain, and run three test questions to verify end-to-end functionality.


In [12]:
# =============================================================================
# CELL 11: BUILD AND TEST THE RAG CHAIN
# =============================================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Prompt instructs the LLM to answer strictly from retrieved context
prompt = ChatPromptTemplate.from_template("""You are a helpful assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say: "I don't know based on the provided documents."
Keep your answer concise and accurate.

Context:
{context}

Question: {question}

Answer:""")

# Assemble the RAG chain using LCEL pipe syntax
# Each | step is automatically traced in LangSmith
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain assembled!")
print("📋 Flow: Question → Retriever → Prompt → LLM → Answer")
print("📊 All steps traced automatically in LangSmith\n")

# Test the complete RAG pipeline
questions = [
    "What is cloud computing?",
    "What are the benefits of cloud computing?",
    "What is data science?"
]

print("🧪 TESTING RAG PIPELINE")
print("=" * 60)

for i, question in enumerate(questions, 1):
    print(f"\n❓ Q{i}: {question}")
    print("-" * 40)
    answer = rag_chain.invoke(question)
    print(f"💡 Answer: {answer}")
    print("=" * 60)

print("\n🎯 RAG testing complete! Check LangSmith for traces.")

✅ RAG chain assembled!
📋 Flow: Question → Retriever → Prompt → LLM → Answer
📊 All steps traced automatically in LangSmith

🧪 TESTING RAG PIPELINE

❓ Q1: What is cloud computing?
----------------------------------------
💡 Answer: Cloud computing is on-demand access to computing resources—physical or virtual servers, data storage, networking capabilities, application development tools, software, AI-powered analytic platforms and more—over the internet with pay-per-use pricing.

❓ Q2: What are the benefits of cloud computing?
----------------------------------------
💡 Answer: The benefits of cloud computing include cost-effectiveness, increased speed and agility, unlimited scalability, and enhanced strategic value.

❓ Q3: What is data science?
----------------------------------------
💡 Answer: Data science combines math and statistics, specialized programming, advanced analytics, artificial intelligence (AI), and machine learning with specific subject matter expertise to uncover actionabl

## Cell 12: Comprehensive Testing
### Why this cell?
Testing with diverse questions helps validate that:
- The retriever finds relevant chunks for different query types
- The LLM generates accurate, context-grounded answers
- The system correctly declines questions outside the document scope

### What we do here:
Run four more questions covering different aspects of the loaded content, including a question the system cannot answer (to verify graceful fallback).


In [13]:
# =============================================================================
# CELL 12: COMPREHENSIVE RAG TESTING
# =============================================================================
print("🧪 Comprehensive RAG System Testing")
print("=" * 70)

test_questions = [
    "What are the key components of cloud computing?",
    "How is data science used in business?",
    "What skills does a data scientist need?",
    "What is the difference between IaaS, PaaS, and SaaS?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{i}. ❓ {question}")
    print("-" * 50)
    try:
        response = rag_chain.invoke(question)
        print(f"   ✅ {response}")
    except Exception as e:
        print(f"   ❌ Error: {e}")
    print()

print("🎉 All questions processed!")
print("📊 View detailed traces at: https://smith.langchain.com → Projects → K21")

🧪 Comprehensive RAG System Testing

1. ❓ What are the key components of cloud computing?
--------------------------------------------------
   ✅ Data centers, networking capabilities, virtualization.


2. ❓ How is data science used in business?
--------------------------------------------------
   ✅ Data science is used in business to uncover actionable insights from an organization’s data through the combination of math and statistics, specialized programming, advanced analytics, AI, and machine learning. These insights guide decision making and strategic planning. Common use cases include process optimization through intelligent automation and enhanced targeting and personalization to improve customer experience (CX).


3. ❓ What skills does a data scientist need?
--------------------------------------------------
   ✅ A data scientist needs computer science and pure science skills, as well as a deep understanding of the specifics of the industry or business discipline they work in. 

## Cell 13: System Summary
### Why this cell?
A summary cell provides a quick health check of all system components, confirming everything is properly configured and operational before handing off to stakeholders or other team members.

### What we do here:
Print a status report for every component in the RAG pipeline.


In [14]:
# =============================================================================
# CELL 13: SYSTEM SUMMARY
# =============================================================================
print("📋 RAG SYSTEM 1 — FINAL SUMMARY")
print("=" * 60)
print("\n🔧 COMPONENTS:")
print(f"   ✅ LangSmith Tracing  : ON — Project: K21")
print(f"   ✅ AWS Bedrock        : Connected (us-east-1)")
print(f"   ✅ LLM                : amazon.nova-lite-v1:0")
print(f"   ✅ Embeddings         : amazon.titan-embed-text-v2:0 (1024-dim)")
print(f"   ✅ Vector Store       : NumpyVectorStore ({vectorstore.count()} chunks)")
print(f"   ✅ Documents Loaded   : {len(docs)} web pages → {len(all_splits)} chunks")
print(f"   ✅ RAG Chain          : LCEL pipeline operational")
print("\n🎯 CAPABILITIES:")
print("   • Answer questions about cloud computing and data science")
print("   • Semantic similarity search over web-loaded documents")
print("   • Context-grounded answers with graceful fallback")
print("\n🚀 SYSTEM READY!")
print("=" * 60)

📋 RAG SYSTEM 1 — FINAL SUMMARY

🔧 COMPONENTS:
   ✅ LangSmith Tracing  : ON — Project: K21
   ✅ AWS Bedrock        : Connected (us-east-1)
   ✅ LLM                : amazon.nova-lite-v1:0
   ✅ Embeddings         : amazon.titan-embed-text-v2:0 (1024-dim)
   ✅ Vector Store       : NumpyVectorStore (169 chunks)
   ✅ Documents Loaded   : 2 web pages → 169 chunks
   ✅ RAG Chain          : LCEL pipeline operational

🎯 CAPABILITIES:
   • Answer questions about cloud computing and data science
   • Semantic similarity search over web-loaded documents
   • Context-grounded answers with graceful fallback

🚀 SYSTEM READY!
